In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency, hmean
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from pandas.api.types import is_numeric_dtype
import statsmodels
from statsmodels.stats.multitest import multipletests

In [ ]:
from general_functions import remove_small_clusters, clinical_enrichment, calculate_stability_metrics, add_normalised_metric, measure_robustness

# Second benchmarking: obtaining the best algorithm

In [ ]:
results1_file = pd.read_csv('benchmarking_files/first_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval, 
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
results2_file = pd.read_csv('benchmarking_files/second_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval,
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)
bench1_2clusters = pd.read_csv('benchmarks_new/firstbench_2clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_3clusters = pd.read_csv('benchmarks_new/firstbench_3clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_4clusters = pd.read_csv('benchmarks_new/firstbench_4clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_5clusters = pd.read_csv('benchmarks_new/firstbench_5clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
frames = [bench1_2clusters, bench1_3clusters, bench1_4clusters, bench1_5clusters]
results1_file = pd.concat(frames)
results2_file = pd.read_csv('benchmarks_new/secondbench.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval,
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
view_combination = results2_file['view_combination'].iloc[0]
n_clusters = results2_file['n_clusters'].iloc[0]
results1_combination = results1_file[(results1_file['n_clusters'] == n_clusters) & (results1_file['view_combination'] == view_combination)]
results2_prov = pd.concat([results1_combination, results2_file])
results2_robustness = measure_robustness(results2_prov)
def substitute_ampmech(df):
    amputation_mechanisms = ['edm', 'pm', 'mnar', 'mcar']
    no_amputation_df = df[df['amputation_mechanism'] == 'No']
    new_rows_list = []
    for mechanism in amputation_mechanisms:
        temp_df = no_amputation_df.copy()
        temp_df['amputation_mechanism'] = mechanism
        new_rows_list.append(temp_df)
    new_amputation_rows = pd.concat(new_rows_list, ignore_index=True)
    df_no_no = df[df['amputation_mechanism'] != 'No']
    df_updated = pd.concat([df_no_no, new_amputation_rows], ignore_index=True)
    return df_updated
results2_df = substitute_ampmech(results2_robustness)

In [ ]:
# Tables for robustness
robustness_20 = results2_df[results2_df['missing_percentage'] == 20]
robustness_df = pd.DataFrame(columns=['Algorithm', 'Best', 'In top 3 models', 'Average rank', 'SD rank'])


In [ ]:
import warnings
warnings.filterwarnings("ignore")
valid_results2, outlier_results2, outlier_patients2 = remove_small_clusters(results2_df, 20, verbose=False)
results_clin2 = clinical_enrichment(valid_results2, clinical_data_file)
normalised_silhouette2 = add_normalised_metric(results_clin2, metric='silhouette', variable_to_normalise='missing_percentage', greater_is_better=True)
stability_metrics_results2 = calculate_stability_metrics(normalised_silhouette2, random_state=42, progress_bar=True)
stability_metrics_results2[['AMI', 'ARI']] = stability_metrics_results2[['AMI','ARI']].fillna(value=0)
stability_metrics_results2[['AMI', 'ARI']] = stability_metrics_results2[['AMI', 'ARI']].clip(lower=0)
normalised_ami2 = add_normalised_metric(stability_metrics_results2, metric='AMI', variable_to_normalise='missing_percentage', greater_is_better=True)
results2 = normalised_ami2.copy()
results2['combined_metric'] = results2[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
results2 = results2.sort_values('combined_metric', ascending=False)
results2['amputation_mechanism'] = results2['amputation_mechanism'].str.replace('edm', 'um')
results2

### Pointplots

In [ ]:
sns.set_theme(style='ticks')

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()

fig, ax = plt.subplots(1, 4, figsize=(18,4))
# Pointplots by algorithm
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0], 'HMean(Adj. Mutual Info., Silhouette)')
plot_pointplots(results2, 'algorithm', 'robustness', ax[1], 'Robustness')
plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[2], 'Log-rank test p-value')
plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[3], 'Mean no. enriched clinical parameters')
handles_alg, labels_alg = ax[0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[3].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
plt.tight_layout(w_pad=3)
plt.savefig('figures/second_bench/algs.svg', bbox_inches='tight')
plt.show()

# Pointplots by amputation mechanism
fig, ax = plt.subplots(1, 4, figsize=(18,4))
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[0], 'HMean(Adj. Mutual Info., Silhouette)')
plot_pointplots(results2, 'amputation_mechanism', 'robustness', ax[1], 'Robustness')
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[2], 'Log-rank test p-value')
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[3], 'Mean no. enriched clinical parameters')
handles_amp, labels_amp = ax[0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[3].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')
plt.tight_layout(w_pad=3)
plt.savefig('figures/second_bench/amp_mech.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()

plt.rcParams.update({'font.size': 12})
fig, ax = plt.subplots(3, 2, figsize=(8,10), sharey = True, sharex = True)
# Pointplots for combined metric
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0, 0])
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[0, 1])
ax[0, 0].set_ylabel('HMean (Adj. Mutual Info., Silhouette)')
# Pointplots for log-rank test p-value
plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[1, 0])
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[1, 1])
ax[1, 0].set_ylabel('Log-rank test p-value')
# Pointplots for enrichment of clinical labels
plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[2, 0])
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[2, 1])
ax[2, 0].set_ylabel('Mean no. enriched clinical parameters')

handles_alg, labels_alg = ax[0, 0].get_legend_handles_labels()
handles_amp, labels_amp = ax[0, 1].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[2, 0].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
ax[2, 1].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')
plt.tight_layout(w_pad=5)
plt.savefig('figures/second_bench/alg_amp_mech_vertical.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()

fig, ax = plt.subplots(2, 3, figsize=(14,7), sharex = True)
# Pointplots by algorithm
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0, 0], 'HMean(Adj. Mutual Info., Silhouette)')
plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[0, 1], 'Log-rank test p-value')
plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[0, 2], 'Mean no. enriched clinical parameters')

# Pointplots by amputation mechanism
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[1, 0], 'HMean(Adj. Mutual Info., Silhouette)')
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[1, 1], 'Log-rank test p-value')
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[1, 2], 'Mean no. enriched clinical parameters')

handles_alg, labels_alg = ax[0, 0].get_legend_handles_labels()
handles_amp, labels_amp = ax[1, 0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[0, 2].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
ax[1, 2].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')
plt.tight_layout(w_pad=5, h_pad=5)
plt.savefig('figures/second_bench/alg_amp_mech_horizontal.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()

fig, ax = plt.subplots(2, 5, figsize=(20,7))
# Pointplots by algorithm
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0, 0], 'HMean(Adj. Mutual Info., Silhouette)')
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[1, 0], 'HMean(Adj. Mutual Info., Silhouette)')

plot_pointplots(results2, 'algorithm', 'normalised_silhouette', ax[0, 1], 'Silhouette (norm.)')
plot_pointplots(results2, 'amputation_mechanism', 'normalised_silhouette', ax[1, 1], 'Silhouette (norm.)')


plot_pointplots(results2, 'algorithm', 'normalised_AMI', ax[0, 2], 'Adj. Mutual Info. (norm.)')
plot_pointplots(results2, 'amputation_mechanism', 'normalised_AMI', ax[1, 2], 'Adj. Mutual Info. (norm.)')

plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[0, 3], 'Log-rank test p-value')
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[1, 3], 'Log-rank test p-value')


plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[0, 4], 'Number of enriched clinical parameters')
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[1, 4], 'Number of enriched clinical parameters')

handles_alg, labels_alg = ax[0, 0].get_legend_handles_labels()
handles_amp, labels_amp = ax[1, 0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[0, 4].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
ax[1, 4].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')

plt.tight_layout(w_pad=5, h_pad=5)
plt.savefig('figures/second_bench/alg_ampmech_complete.svg', bbox_inches='tight')
plt.show()